# Advanced Generator-Based Context Managers — Problems with Complete Solutions

This notebook develops production-quality context managers with:

- `contextlib.contextmanager`
- generator functions and `yield`
- exception propagation, translation, and selective suppression
- deterministic cleanup
- nested and dynamically acquired resources
- `ExitStack`
- `ContextDecorator`
- `contextvars`
- `asynccontextmanager`
- a simplified implementation of the generator-context-manager protocol

Every problem includes a specification, constraints, a complete solution, tests, and design notes.

> Target runtime: Python 3.10+

## Learning goals

By the end, you should be able to:

1. Explain exactly what happens before, at, and after `yield`.
2. Guarantee cleanup with `try/finally`.
3. Correctly handle exceptions raised inside a `with` block.
4. Decide when an exception should propagate, be translated, or be suppressed.
5. Compose a variable number of resources safely with `ExitStack`.
6. Avoid common mistakes such as yielding twice or reusing a one-shot manager.
7. Build synchronous and asynchronous context managers using standard-library tools.

## Best-practice checklist

Use these rules throughout the notebook:

1. Put cleanup in `finally`.
2. Acquire the resource before `yield`; release it after `yield`.
3. Yield exactly once.
4. Do not catch `BaseException` unless you have a very specific reason.
5. Suppress only exceptions you intentionally own and understand.
6. Restore the previous state, not an assumed default state.
7. Prefer standard-library managers when one already exists.
8. Use `ExitStack` for a dynamic number of resources.
9. Treat each generator-based context-manager instance as one-shot.
10. Keep the manager focused on resource lifetime; keep business logic outside it.

## Shared imports

In [1]:
from __future__ import annotations

import asyncio
import contextvars
import io
import logging
import os
import random
import shutil
import sys
import tempfile
import time
import uuid

from contextlib import (
    ExitStack,
    ContextDecorator,
    asynccontextmanager,
    contextmanager,
    nullcontext,
    redirect_stderr,
    redirect_stdout,
)
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Generator, Iterable, Iterator, TextIO

## Core mental model

A generator-based context manager has three phases:

```python
@contextmanager
def manager():
    acquire()
    try:
        resource = build_resource()
        yield resource       # value bound by "as"
    finally:
        release()
```

Conceptually:

- `__enter__()` advances the generator to `yield`.
- Normal exit resumes immediately after `yield`.
- Exceptional exit injects the exception at the `yield` expression.
- Cleanup must still happen.
- If the generator finishes without re-raising the injected exception, the exception is suppressed.

# Problem 1 — Exception-safe benchmarking with structured results

Create `benchmark(label)`.

Requirements:

- Measure elapsed wall-clock time using `time.perf_counter()`.
- Yield a mutable result object.
- Record whether the block succeeded.
- Record the exception type without suppressing the exception.
- Always populate the ending timestamp and elapsed duration.
- Reject an empty label.

In [2]:
@dataclass
class BenchmarkResult:
    label: str
    started_at: float
    finished_at: float | None = None
    elapsed_seconds: float | None = None
    succeeded: bool = False
    exception_type: str | None = None


@contextmanager
def benchmark(label: str) -> Iterator[BenchmarkResult]:
    if not label or not label.strip():
        raise ValueError("label must be a non-empty string")

    result = BenchmarkResult(
        label=label.strip(),
        started_at=time.perf_counter(),
    )

    try:
        yield result
    except Exception as exc:
        result.exception_type = type(exc).__name__
        raise
    else:
        result.succeeded = True
    finally:
        result.finished_at = time.perf_counter()
        result.elapsed_seconds = result.finished_at - result.started_at

In [3]:
with benchmark("small calculation") as successful:
    total = sum(i * i for i in range(10_000))

assert successful.succeeded is True
assert successful.exception_type is None
assert successful.elapsed_seconds is not None
assert successful.elapsed_seconds >= 0

try:
    with benchmark("failing operation") as failed:
        raise RuntimeError("boom")
except RuntimeError:
    pass

assert failed.succeeded is False
assert failed.exception_type == "RuntimeError"
assert failed.elapsed_seconds is not None

successful, failed

(BenchmarkResult(label='small calculation', started_at=3843512.6517085, finished_at=3843512.6534595, elapsed_seconds=0.0017510000616312027, succeeded=True, exception_type=None),
 BenchmarkResult(label='failing operation', started_at=3843512.6540356, finished_at=3843512.6540867, elapsed_seconds=5.110027268528938e-05, succeeded=False, exception_type='RuntimeError'))

### Why this design is robust

- `else` runs only when the managed block succeeds.
- `except` records metadata and then re-raises.
- `finally` runs in both cases.
- The result object is mutable, so data written after `yield` remains visible to the caller.

# Problem 2 — Temporary environment variables with exact restoration

Implement `temporary_environment(updates, remove=())`.

Requirements:

- Set or replace keys from `updates`.
- Temporarily remove keys named in `remove`.
- Restore the exact previous environment afterward.
- Correctly distinguish “missing before entry” from “present with an empty value”.
- Reject a key that appears in both `updates` and `remove`.
- Restore state even when the block fails.

In [4]:
_MISSING = object()


@contextmanager
def temporary_environment(
    updates: dict[str, str],
    remove: Iterable[str] = (),
) -> Iterator[None]:
    remove_set = set(remove)
    overlap = set(updates) & remove_set
    if overlap:
        raise ValueError(
            f"Keys cannot be both updated and removed: {sorted(overlap)}"
        )

    affected = set(updates) | remove_set
    previous: dict[str, str | object] = {
        key: os.environ.get(key, _MISSING)
        for key in affected
    }

    try:
        for key, value in updates.items():
            if not isinstance(key, str) or not isinstance(value, str):
                raise TypeError("environment keys and values must be strings")
            os.environ[key] = value

        for key in remove_set:
            os.environ.pop(key, None)

        yield
    finally:
        for key, old_value in previous.items():
            if old_value is _MISSING:
                os.environ.pop(key, None)
            else:
                os.environ[key] = old_value

In [5]:
key_a = "CTX_DEMO_A"
key_b = "CTX_DEMO_B"

os.environ[key_a] = "original"
os.environ.pop(key_b, None)

try:
    with temporary_environment({key_a: "temporary", key_b: ""}):
        assert os.environ[key_a] == "temporary"
        assert os.environ[key_b] == ""
        raise LookupError("force cleanup")
except LookupError:
    pass

assert os.environ[key_a] == "original"
assert key_b not in os.environ

print("Environment restored exactly.")

Environment restored exactly.


### Design lesson

Never restore a global setting to an assumed default. Save the previous value and restore that exact value.

# Problem 3 — Atomic text-file replacement

Implement `atomic_text_writer(destination, encoding="utf-8")`.

The caller writes to a temporary file. On successful exit, replace the destination atomically with `os.replace`. On failure, remove the temporary file and leave the original destination unchanged.

Requirements:

- Create the temporary file in the destination directory.
- Flush and `fsync` before replacement.
- Preserve the original file if the block raises.
- Clean up the temporary file in all cases.
- Yield an open text file object.

In [6]:
@contextmanager
def atomic_text_writer(
    destination: str | os.PathLike[str],
    *,
    encoding: str = "utf-8",
) -> Iterator[TextIO]:
    destination_path = Path(destination)
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    fd, temp_name = tempfile.mkstemp(
        prefix=f".{destination_path.name}.",
        suffix=".tmp",
        dir=destination_path.parent,
        text=True,
    )
    temp_path = Path(temp_name)
    file_obj: TextIO | None = None

    try:
        file_obj = os.fdopen(fd, mode="w", encoding=encoding, newline="")
        fd = -1

        try:
            yield file_obj
            file_obj.flush()
            os.fsync(file_obj.fileno())
        finally:
            file_obj.close()

        os.replace(temp_path, destination_path)
    finally:
        if file_obj is not None and not file_obj.closed:
            file_obj.close()
        if fd != -1:
            os.close(fd)
        temp_path.unlink(missing_ok=True)

In [7]:
with tempfile.TemporaryDirectory() as temp_dir:
    target = Path(temp_dir) / "report.txt"
    target.write_text("old content", encoding="utf-8")

    with atomic_text_writer(target) as stream:
        stream.write("new content")

    assert target.read_text(encoding="utf-8") == "new content"

    try:
        with atomic_text_writer(target) as stream:
            stream.write("partial and invalid")
            raise ValueError("validation failed")
    except ValueError:
        pass

    assert target.read_text(encoding="utf-8") == "new content"

print("Atomic write behavior verified.")

Atomic write behavior verified.


### Production note

`os.replace` is atomic when source and destination are on the same filesystem. Creating the temporary file in the destination directory is therefore important.

# Problem 4 — Transaction manager with commit, rollback, and optional suppression

Build a transaction context manager for a small fake connection object.

Behavior:

- Entering begins a transaction.
- Normal exit commits.
- Exceptional exit rolls back.
- By default, the original exception propagates.
- If `suppress=(SomeError,)` is supplied, only matching exceptions are suppressed.
- Commit failures must propagate and must not be mislabeled as body failures.

In [8]:
@dataclass
class FakeConnection:
    events: list[str] = field(default_factory=list)

    def begin(self) -> None:
        self.events.append("BEGIN")

    def commit(self) -> None:
        self.events.append("COMMIT")

    def rollback(self) -> None:
        self.events.append("ROLLBACK")


@contextmanager
def transaction(
    connection: FakeConnection,
    *,
    suppress: tuple[type[BaseException], ...] = (),
) -> Iterator[FakeConnection]:
    connection.begin()

    try:
        yield connection
    except Exception as exc:
        connection.rollback()
        if isinstance(exc, suppress):
            return
        raise
    else:
        connection.commit()

In [9]:
conn = FakeConnection()
with transaction(conn):
    conn.events.append("WRITE")
assert conn.events == ["BEGIN", "WRITE", "COMMIT"]

conn = FakeConnection()
try:
    with transaction(conn):
        conn.events.append("WRITE")
        raise ValueError("invalid")
except ValueError:
    pass
assert conn.events == ["BEGIN", "WRITE", "ROLLBACK"]

conn = FakeConnection()
with transaction(conn, suppress=(LookupError,)):
    conn.events.append("READ")
    raise KeyError("optional row missing")
assert conn.events == ["BEGIN", "READ", "ROLLBACK"]

print("Transaction semantics verified.")

Transaction semantics verified.


### Best practice

Suppression should be narrow and explicit. A broad `except Exception: return` can hide programming errors and corrupt control flow.

# Problem 5 — Translate a low-level exception while preserving causality

Create `translate_os_errors(operation)`.

Requirements:

- Convert `OSError` raised by the managed block into `ResourceOperationError`.
- Include the operation name.
- Preserve the original exception as `__cause__`.
- Do not translate unrelated exceptions.
- Do not suppress anything.

In [10]:
class ResourceOperationError(RuntimeError):
    pass


@contextmanager
def translate_os_errors(operation: str) -> Iterator[None]:
    try:
        yield
    except OSError as exc:
        raise ResourceOperationError(
            f"{operation} failed: {exc}"
        ) from exc

In [11]:
try:
    with translate_os_errors("load configuration"):
        raise FileNotFoundError("settings.toml")
except ResourceOperationError as exc:
    assert isinstance(exc.__cause__, FileNotFoundError)
    print(type(exc).__name__, "->", type(exc.__cause__).__name__)

try:
    with translate_os_errors("compute"):
        raise ValueError("not an OS error")
except ValueError:
    print("Unrelated exception propagated unchanged.")

ResourceOperationError -> FileNotFoundError
Unrelated exception propagated unchanged.


# Problem 6 — Temporarily patch an object attribute

Implement `temporary_attribute(obj, name, value)`.

Requirements:

- Handle an attribute that did not previously exist.
- Restore an existing attribute exactly.
- Delete a newly introduced attribute on exit.
- Work correctly when the managed block raises.

In [12]:
@contextmanager
def temporary_attribute(
    obj: Any,
    name: str,
    value: Any,
) -> Iterator[Any]:
    previous = getattr(obj, name, _MISSING)
    setattr(obj, name, value)

    try:
        yield obj
    finally:
        if previous is _MISSING:
            delattr(obj, name)
        else:
            setattr(obj, name, previous)

In [13]:
class Settings:
    mode = "production"


settings = Settings()

with temporary_attribute(settings, "mode", "testing"):
    assert settings.mode == "testing"
assert settings.mode == "production"

assert not hasattr(settings, "feature_flag")
try:
    with temporary_attribute(settings, "feature_flag", True):
        assert settings.feature_flag is True
        raise RuntimeError("test cleanup")
except RuntimeError:
    pass
assert not hasattr(settings, "feature_flag")

print("Attribute restoration verified.")

Attribute restoration verified.


# Problem 7 — Dynamic resource acquisition with `ExitStack`

Create `open_text_files(paths)`.

Requirements:

- Accept any iterable of paths.
- Open every file for reading.
- Return a list of file objects.
- If opening file number *n* fails, close files `0..n-1`.
- Close all files on normal and exceptional exit.
- Avoid manually writing nested `with` statements.

In [14]:
@contextmanager
def open_text_files(
    paths: Iterable[str | os.PathLike[str]],
    *,
    encoding: str = "utf-8",
) -> Iterator[list[TextIO]]:
    with ExitStack() as stack:
        files = [
            stack.enter_context(open(path, "r", encoding=encoding))
            for path in paths
        ]
        yield files

In [15]:
with tempfile.TemporaryDirectory() as temp_dir:
    root = Path(temp_dir)
    paths = []

    for index in range(4):
        path = root / f"part-{index}.txt"
        path.write_text(f"line-{index}\n", encoding="utf-8")
        paths.append(path)

    with open_text_files(paths) as streams:
        assert [stream.read().strip() for stream in streams] == [
            "line-0", "line-1", "line-2", "line-3"
        ]
        assert all(not stream.closed for stream in streams)

    assert all(stream.closed for stream in streams)

print("Dynamic resource cleanup verified.")

Dynamic resource cleanup verified.


### Why `ExitStack` matters

A normal multi-item `with` statement works only when the number of resources is known while writing the code. `ExitStack` handles a number known only at runtime and unwinds already-entered resources if a later acquisition fails.

# Problem 8 — Capture both stdout and stderr

Create `capture_output()` that yields an object containing two `StringIO` buffers.

Requirements:

- Capture `print(...)` output.
- Capture writes to `sys.stderr`.
- Restore both streams even after failure.
- Use standard-library redirection managers instead of mutating `sys.stdout` and `sys.stderr` manually.

In [16]:
@dataclass
class CapturedOutput:
    stdout: io.StringIO
    stderr: io.StringIO


@contextmanager
def capture_output() -> Iterator[CapturedOutput]:
    captured = CapturedOutput(
        stdout=io.StringIO(),
        stderr=io.StringIO(),
    )

    with redirect_stdout(captured.stdout), redirect_stderr(captured.stderr):
        yield captured

In [17]:
with capture_output() as captured:
    print("normal message")
    print("warning message", file=sys.stderr)

assert captured.stdout.getvalue() == "normal message\n"
assert captured.stderr.getvalue() == "warning message\n"

print("stdout:", captured.stdout.getvalue().strip())
print("stderr:", captured.stderr.getvalue().strip())

stdout: normal message
stderr: warning message


# Problem 9 — Temporary workspace and working-directory restoration

Create `temporary_workspace()`.

Requirements:

- Create a temporary directory.
- Change the process working directory to it.
- Yield the workspace `Path`.
- Restore the exact previous working directory before deleting the workspace.
- Delete the workspace on all exit paths.
- Explain why changing the working directory is process-global and therefore unsafe in many threaded programs.

In [18]:
@contextmanager
def temporary_workspace(
    *,
    prefix: str = "ctx-workspace-",
) -> Iterator[Path]:
    previous_cwd = Path.cwd()
    workspace = Path(tempfile.mkdtemp(prefix=prefix))

    try:
        os.chdir(workspace)
        yield workspace
    finally:
        os.chdir(previous_cwd)
        shutil.rmtree(workspace, ignore_errors=False)

In [19]:
original = Path.cwd()

with temporary_workspace() as workspace:
    assert Path.cwd() == workspace
    Path("artifact.txt").write_text("temporary", encoding="utf-8")
    assert (workspace / "artifact.txt").exists()

assert Path.cwd() == original
assert not workspace.exists()

print("Working directory and workspace restored.")

Working directory and workspace restored.


### Concurrency warning

`os.chdir` changes process-wide state, not task-local or thread-local state. Prefer absolute paths in libraries and concurrent applications.

# Problem 10 — Correlation IDs using `contextvars`

Create `correlation_scope(correlation_id=None)`.

Requirements:

- If no ID is supplied, generate one.
- Make the active ID available through a `ContextVar`.
- Support correct nesting.
- Restore the previous value with the token returned by `ContextVar.set`.
- Yield the active ID.

In [20]:
current_correlation_id: contextvars.ContextVar[str | None] = (
    contextvars.ContextVar("current_correlation_id", default=None)
)


@contextmanager
def correlation_scope(
    correlation_id: str | None = None,
) -> Iterator[str]:
    active_id = correlation_id or uuid.uuid4().hex
    token = current_correlation_id.set(active_id)

    try:
        yield active_id
    finally:
        current_correlation_id.reset(token)

In [21]:
assert current_correlation_id.get() is None

with correlation_scope("outer") as outer:
    assert outer == "outer"
    assert current_correlation_id.get() == "outer"

    with correlation_scope("inner") as inner:
        assert inner == "inner"
        assert current_correlation_id.get() == "inner"

    assert current_correlation_id.get() == "outer"

assert current_correlation_id.get() is None

print("Nested context-local state restored correctly.")

Nested context-local state restored correctly.


### Why not use a module-level global variable?

A plain global cannot safely represent independent values for nested asynchronous tasks. `ContextVar` is designed for context-local state.

# Problem 11 — A context manager that is also a decorator

Use `ContextDecorator` to build `temporary_log_level(logger, level)`.

Requirements:

- Work with both `with` and `@decorator`.
- Restore the exact previous level.
- Preserve behavior under exceptions.
- Demonstrate both forms.

In [22]:
class temporary_log_level(ContextDecorator):
    def __init__(self, logger: logging.Logger, level: int) -> None:
        self.logger = logger
        self.level = level
        self._previous_level: int | None = None

    def __enter__(self) -> logging.Logger:
        self._previous_level = self.logger.level
        self.logger.setLevel(self.level)
        return self.logger

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc: BaseException | None,
        traceback: Any,
    ) -> bool:
        assert self._previous_level is not None
        self.logger.setLevel(self._previous_level)
        return False

In [23]:
demo_logger = logging.getLogger("context-manager-demo")
demo_logger.setLevel(logging.WARNING)

with temporary_log_level(demo_logger, logging.DEBUG):
    assert demo_logger.level == logging.DEBUG
assert demo_logger.level == logging.WARNING


@temporary_log_level(demo_logger, logging.INFO)
def decorated_operation() -> int:
    assert demo_logger.level == logging.INFO
    return 42


assert decorated_operation() == 42
assert demo_logger.level == logging.WARNING

print("ContextDecorator works in both forms.")

ContextDecorator works in both forms.


### Caveat

A decorator instance may be reused across calls. Therefore, a `ContextDecorator` class should not store per-entry state in a way that breaks nested or concurrent use. The simple example above is educational; a production version should create independent state per entry or document non-reentrancy.

# Problem 12 — Understand one-shot manager instances

A function decorated with `@contextmanager` is reusable, but each object returned by calling it is a one-shot context-manager instance.

Demonstrate the failure, then write a safe factory-based use pattern.

In [24]:
@contextmanager
def labeled_scope(label: str) -> Iterator[str]:
    print(f"enter {label}")
    try:
        yield label
    finally:
        print(f"exit {label}")

In [25]:
manager_instance = labeled_scope("one-shot")

with manager_instance:
    pass

reuse_error: Exception | None = None
try:
    with manager_instance:
        pass
except Exception as exc:
    reuse_error = exc

assert reuse_error is not None
print("Reusing one instance fails with:", type(reuse_error).__name__)

for _ in range(2):
    with labeled_scope("fresh-instance"):
        pass

enter one-shot
exit one-shot
Reusing one instance fails with: AttributeError
enter fresh-instance
exit fresh-instance
enter fresh-instance
exit fresh-instance


### Rule

Store the factory when reuse is needed:

```python
scope_factory = lambda: labeled_scope("new each time")
with scope_factory():
    ...
with scope_factory():
    ...
```

Do not store and reuse a single generator-context-manager instance.

# Problem 13 — Implement a simplified generator context-manager adapter

Implement enough of the protocol to understand what `contextlib.contextmanager` does.

Your adapter must:

- Advance to the first `yield` on entry.
- Raise an error if the generator never yields.
- On normal exit, require the generator to stop after the first yield.
- On exceptional exit, inject the exception with `generator.throw(...)`.
- Suppress the exception only if the generator handles it and terminates.
- Raise an error if the generator yields a second time.

This is educational. Use `contextlib.contextmanager` in real projects.

In [26]:
class SimpleGeneratorContextManager:
    def __init__(
        self,
        generator_factory: Callable[..., Generator[Any, None, None]],
        args: tuple[Any, ...],
        kwargs: dict[str, Any],
    ) -> None:
        self._generator = generator_factory(*args, **kwargs)

    def __enter__(self) -> Any:
        try:
            return next(self._generator)
        except StopIteration as exc:
            raise RuntimeError("generator did not yield") from exc

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc: BaseException | None,
        traceback: Any,
    ) -> bool:
        if exc_type is None:
            try:
                next(self._generator)
            except StopIteration:
                return False
            else:
                try:
                    self._generator.close()
                finally:
                    raise RuntimeError(
                        "generator did not stop after yield"
                    )

        assert exc is not None

        try:
            self._generator.throw(exc_type, exc, traceback)
        except StopIteration:
            return True
        except BaseException as raised:
            if raised is exc:
                return False
            raise
        else:
            try:
                self._generator.close()
            finally:
                raise RuntimeError(
                    "generator did not stop after handling exception"
                )


def simple_contextmanager(
    generator_factory: Callable[..., Generator[Any, None, None]],
) -> Callable[..., SimpleGeneratorContextManager]:
    def helper(*args: Any, **kwargs: Any) -> SimpleGeneratorContextManager:
        return SimpleGeneratorContextManager(
            generator_factory,
            args,
            kwargs,
        )

    return helper

In [27]:
@simple_contextmanager
def demo_manager(events: list[str]) -> Iterator[str]:
    events.append("acquire")
    try:
        yield "resource"
    finally:
        events.append("release")


events: list[str] = []
with demo_manager(events) as resource:
    assert resource == "resource"
    events.append("use")

assert events == ["acquire", "use", "release"]


@simple_contextmanager
def suppress_key_error() -> Iterator[None]:
    try:
        yield
    except KeyError:
        pass


with suppress_key_error():
    raise KeyError("intentionally suppressed")


@simple_contextmanager
def propagate_value_error() -> Iterator[None]:
    yield


try:
    with propagate_value_error():
        raise ValueError("must propagate")
except ValueError:
    pass
else:
    raise AssertionError("ValueError should have propagated")

print("Simplified adapter passed its core tests.")

Simplified adapter passed its core tests.


C:\Users\user1\AppData\Local\Temp\ipykernel_9164\3281096935.py:38: DeprecationWarning: the (type, exc, tb) signature of throw() is deprecated, use the single-arg signature instead.
  self._generator.throw(exc_type, exc, traceback)


### Important subtlety

The real implementation in `contextlib` handles additional edge cases, traceback identity details, generator cleanup, and exception-group behavior. Reimplementing it is useful for learning, not for production.

# Problem 14 — Asynchronous context manager with `asynccontextmanager`

Build `async_connection(name)`.

Requirements:

- Simulate asynchronous connection setup and teardown.
- Yield a connection object.
- Always close it.
- Demonstrate use with `async with`.
- Show that cleanup still runs when the body raises.

In [28]:
@dataclass
class AsyncConnection:
    name: str
    closed: bool = False

    async def execute(self, query: str) -> str:
        if self.closed:
            raise RuntimeError("connection is closed")
        await asyncio.sleep(0)
        return f"{self.name}: {query}"


@asynccontextmanager
async def async_connection(name: str) -> Any:
    await asyncio.sleep(0)
    connection = AsyncConnection(name=name)

    try:
        yield connection
    finally:
        await asyncio.sleep(0)
        connection.closed = True

In [29]:
async def async_demo() -> None:
    async with async_connection("primary") as connection:
        result = await connection.execute("SELECT 1")
        assert result == "primary: SELECT 1"

    assert connection.closed is True

    try:
        async with async_connection("secondary") as failing:
            raise RuntimeError("body failure")
    except RuntimeError:
        pass

    assert failing.closed is True


await async_demo()
print("Async cleanup verified.")

Async cleanup verified.


# Additional solved drills

The following shorter problems add breadth and more code patterns.

## Drill A — Preserve and restore the random generator state

Useful in tests that require deterministic random values without permanently disturbing global random state.

In [30]:
@contextmanager
def preserved_random_state(seed: int | None = None) -> Iterator[None]:
    state = random.getstate()
    try:
        if seed is not None:
            random.seed(seed)
        yield
    finally:
        random.setstate(state)

In [31]:
random.seed(100)
before = random.random()

with preserved_random_state(seed=7):
    inside_first = random.random()
    inside_second = random.random()

after = random.random()

random.seed(100)
assert before == random.random()
assert after == random.random()

random.seed(7)
assert inside_first == random.random()
assert inside_second == random.random()

print("Random state restored.")

Random state restored.


## Drill B — Register arbitrary cleanup callbacks

Use `ExitStack.callback` to guarantee LIFO cleanup even for APIs that are not context managers.

In [32]:
@contextmanager
def callback_scope() -> Iterator[Callable[..., None]]:
    with ExitStack() as stack:
        def register(
            callback: Callable[..., Any],
            *args: Any,
            **kwargs: Any,
        ) -> None:
            stack.callback(callback, *args, **kwargs)

        yield register

In [33]:
events: list[str] = []

with callback_scope() as register:
    register(events.append, "cleanup-1")
    register(events.append, "cleanup-2")
    events.append("body")

assert events == ["body", "cleanup-2", "cleanup-1"]
print(events)

['body', 'cleanup-2', 'cleanup-1']


## Drill C — Conditional context management

Sometimes a resource is already open. Use `nullcontext` for a no-op manager.

In [34]:
def read_text_source(
    source: str | os.PathLike[str] | TextIO,
    *,
    encoding: str = "utf-8",
) -> str:
    manager = (
        open(source, "r", encoding=encoding)
        if isinstance(source, (str, os.PathLike))
        else nullcontext(source)
    )

    with manager as stream:
        return stream.read()

In [35]:
with tempfile.TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "data.txt"
    path.write_text("from path", encoding="utf-8")

    assert read_text_source(path) == "from path"

    existing_stream = io.StringIO("from stream")
    assert read_text_source(existing_stream) == "from stream"
    assert existing_stream.closed is False

print("Conditional ownership handled correctly.")

Conditional ownership handled correctly.


# Failure-mode laboratory

These examples intentionally demonstrate invalid generator context managers.

In [36]:
@contextmanager
def never_yields() -> Iterator[None]:
    if False:
        yield


@contextmanager
def yields_twice() -> Iterator[int]:
    yield 1
    yield 2

In [37]:
errors: list[str] = []

try:
    with never_yields():
        pass
except RuntimeError as exc:
    errors.append(type(exc).__name__ + ": " + str(exc))

try:
    with yields_twice():
        pass
except RuntimeError as exc:
    errors.append(type(exc).__name__ + ": " + str(exc))

for error in errors:
    print(error)

RuntimeError: generator didn't yield
RuntimeError: generator didn't stop


# Comprehensive review questions with answers

### 1. Why must cleanup usually be in `finally`?

Because code after `yield` is not guaranteed to run normally. An exception from the managed block is injected at `yield`; `finally` guarantees cleanup during both normal and exceptional unwinding.

### 2. What value is bound after `as`?

The value yielded by the generator.

### 3. How can a generator-based manager suppress an exception?

Catch the injected exception and terminate without re-raising it. Suppression should be deliberate and narrow.

### 4. Why should the manager yield exactly once?

The context-manager protocol has one entry and one exit. No yield means entry fails; a second yield means exit cannot complete correctly.

### 5. When should `ExitStack` be preferred?

When the number of resources is dynamic, acquisition is conditional, cleanup callbacks must be registered, or partial acquisition must unwind safely.

### 6. Is a generator-context-manager instance reusable?

No. The decorated function is a reusable factory, but each returned manager instance wraps a one-shot generator.

### 7. Why restore previous state rather than a default?

The caller may already be inside another temporary override. Exact restoration makes nesting work.

### 8. When is `asynccontextmanager` necessary?

When setup or cleanup must `await`, such as acquiring an async lock, opening an async connection, or closing an asynchronous client.

# Final challenge — Compose several managers safely

The solution below combines:

- a temporary environment
- a temporary workspace
- output capture
- benchmarking
- atomic output writing

Observe that each concern remains isolated in its own context manager.

In [38]:
def run_demo_pipeline() -> dict[str, Any]:
    with benchmark("demo pipeline") as stats:
        with temporary_environment({"PIPELINE_MODE": "test"}):
            with temporary_workspace() as workspace:
                with capture_output() as captured:
                    print("pipeline starting")
                    print(f"mode={os.environ['PIPELINE_MODE']}")

                    destination = workspace / "result.txt"
                    with atomic_text_writer(destination) as stream:
                        stream.write("completed\n")

                    result_text = destination.read_text(
                        encoding="utf-8"
                    )

    return {
        "succeeded": stats.succeeded,
        "elapsed_seconds": stats.elapsed_seconds,
        "stdout": captured.stdout.getvalue(),
        "result_text": result_text,
        "workspace_removed": not workspace.exists(),
    }


pipeline_result = run_demo_pipeline()

assert pipeline_result["succeeded"] is True
assert "pipeline starting" in pipeline_result["stdout"]
assert pipeline_result["result_text"] == "completed\n"
assert pipeline_result["workspace_removed"] is True

pipeline_result

{'succeeded': True,
 'elapsed_seconds': 0.019895900040864944,
 'stdout': 'pipeline starting\nmode=test\n',
 'result_text': 'completed\n',
 'workspace_removed': True}

# Summary

Production-quality generator context managers are small state machines:

1. Validate inputs.
2. Acquire state or resources.
3. `yield` exactly once.
4. Handle only expected exceptions.
5. Restore or release everything in `finally`.
6. Compose managers instead of building one giant manager.
7. Prefer `contextlib` utilities over custom protocol implementations.

The strongest designs are easy to test on both normal and exceptional paths.